In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Parameters
# =========================
g = 1.0   # transverse field strength
J = 1.0   # Ising coupling strength

# =========================
# Hamiltonian matrix H(s)
# =========================
I = np.eye(2)
sx = np.array([[0, 1],[1, 0]])
sz = np.array([[1, 0],[0, -1]])

def A(s):
    return s

def B(s):
    return 1-s

def Hamiltonian(s, g=1, J=1):
    HI =  np.kron(I, sx) + np.kron(sx, I)
    HF =  np.kron(sz, sz) + np.kron(sz, I) + np.kron(I, sz)
    return -g*A(s)*HI + -J*B(s)*HF


# =========================
# Annealing grid
# =========================
s_vals = np.linspace(0, 1, 300)

# Store eigenvalues
eigenvalues = np.zeros((len(s_vals), 4))

# Diagonalise at each s
for i, s in enumerate(s_vals):
    w = np.linalg.eigvalsh(Hamiltonian(s, g, J))  # sorted eigenvalues
    eigenvalues[i] = w

# =========================
# Plot eigenvalues
# =========================
plt.figure(figsize=(8,6))

for n in range(4):
    plt.plot(s_vals, eigenvalues[:, n], label=f"Eigenvalue {n+1}", linewidth=2)

plt.xlabel("Annealing parameter s", fontsize=12)
plt.ylabel("Energy eigenvalues", fontsize=12)
plt.title(f"Adiabatic spectrum of H(s) for g = {g}, J = {J}", fontsize=14)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
gap = eigenvalues[:,1] - eigenvalues[:,0]

plt.figure(figsize=(8,6))
plt.plot(s_vals, gap, linewidth=2)
plt.xlabel("s")
plt.ylabel("Energy gap Δ(s)")
plt.title("Instantaneous energy gap")
plt.grid(True)
plt.show()

print("Minimum gap =", np.min(gap))
print("Occurs at s =", s_vals[np.argmin(gap)])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load REAL D-Wave schedule
file_path = "/content/09-1323A-A_Advantage2_system4_1_annealing_schedule.xlsx"
schedule = pd.read_excel(file_path, sheet_name="Standard-Annealing Schedule")

s_vals = schedule["s"].values
A_vals = schedule["A(s) (GHz)"].values
B_vals = schedule["B(s) (GHz)"].values

# plt.figure(figsize=(10, 6))

plt.plot(s_vals, A_vals, linewidth=2.5, label="A(s) : Transverse Field", color="red")
plt.plot(s_vals, B_vals, linewidth=2.5, label="B(s) : Problem Scale", color="blue")

plt.xlabel("Anneal Fraction  s", fontsize=13)
plt.ylabel("Energy Scale (GHz)", fontsize=13)
plt.title("D-Wave Advantage2 Annealing Schedule", fontsize=15, fontweight="bold")

plt.legend(frameon=True, fontsize=11)
# plt.grid(True, which='both', linestyle='--', alpha=0.6)

plt.minorticks_on()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================
# Load D-Wave schedule
# =============================
file_path = "/content/09-1323A-A_Advantage2_system4_1_annealing_schedule.xlsx"
schedule = pd.read_excel(file_path, sheet_name="Standard-Annealing Schedule")

s_vals = schedule["s"].values
A_vals = schedule["A(s) (GHz)"].values
B_vals = schedule["B(s) (GHz)"].values

# =============================
# Polynomial degree
# =============================
degree = 4

# =============================
# Polynomial fitting
# =============================
coeffs_A = np.polyfit(s_vals, A_vals, degree)
coeffs_B = np.polyfit(s_vals, B_vals, degree)

A_poly = np.poly1d(coeffs_A)
B_poly = np.poly1d(coeffs_B)

# =============================
# Final reusable functions
# =============================
def dwaveA(s):
    return A_poly(s)

def dwaveB(s):
    return B_poly(s)

A_fit = dwaveA(s_vals)
B_fit = dwaveB(s_vals)

# =============================
# Goodness-of-fit metrics
# =============================
def fit_metrics(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - ss_res/ss_tot
    return rmse, r2

A_rmse, A_r2 = fit_metrics(A_vals, A_fit)
B_rmse, B_r2 = fit_metrics(B_vals, B_fit)

# =============================
# SUBPLOTS WITH DIFFERENT COLORS
# =============================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---- A(s) subplot ----
axes[0].scatter(s_vals, A_vals, color="darkorange", s=15, label="Raw A(s)")
axes[0].plot(s_vals, A_fit, color="red", linewidth=1.5, label="Polynomial Fit")
axes[0].set_title("D-Wave A(s) Polynomial Fit", fontweight="bold")
axes[0].set_xlabel("Anneal fraction s")
axes[0].set_ylabel("A(s) (GHz)")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# ---- B(s) subplot ----
axes[1].scatter(s_vals, B_vals, color="royalblue", s=15, label="Raw B(s)")
axes[1].plot(s_vals, B_fit, color="navy", linewidth=1.5, label="Polynomial Fit")
axes[1].set_title("D-Wave B(s) Polynomial Fit", fontweight="bold")
axes[1].set_xlabel("Anneal fraction s")
axes[1].set_ylabel("B(s) (GHz)")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.suptitle("Polynomial Approximation of D-Wave Annealing Schedules", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# =============================
# Print symbolic functional forms
# =============================
print("\n------ FINAL FUNCTIONS ------")
print("dwaveA(s) =", A_poly)
print("dwaveB(s) =", B_poly)

print("\n--- Goodness of Fit ---")
print(f"A(s): RMSE = {A_rmse:.6f},   R^2 = {A_r2:.8f}")
print(f"B(s): RMSE = {B_rmse:.6f},   R^2 = {B_r2:.8f}")



In [ ]:

# =============================
# Residuals (Subplots + Different Colors)
# =============================

A_residual = A_vals - A_fit
B_residual = B_vals - B_fit

fig, axes = plt.subplots(1, 2, figsize=(12,5))

# ---- Residual for A(s) ----
axes[0].plot(
    s_vals,
    A_residual,
    color="green",
    linestyle="-",
    linewidth=2,
    label="Residual A(s)"
)
axes[0].axhline(0, color="black", linestyle=":", linewidth=1.5)
axes[0].set_title("Residual Error for A(s)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Anneal fraction s")
axes[0].set_ylabel("Residual Error (GHz)")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# ---- Residual for B(s) ----
axes[1].plot(
    s_vals,
    B_residual,
    color="blue",
    linestyle="-",
    linewidth=2,
    label="Residual B(s)"
)
axes[1].axhline(0, color="black", linestyle=":", linewidth=1.5)
axes[1].set_title("Residual Error for B(s)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Anneal fraction s")
axes[1].set_ylabel("Residual Error (GHz)")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.suptitle("Residual Errors of Polynomial Fits for D-Wave Schedules",
             fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:
df = pd.DataFrame({
    "s": s_vals,
    "A(s) (GHz)": A_vals,
    "B(s) (GHz)": B_vals
})

df.describe()

In [ ]:
g = 1
J = 1


# =========================
# Hamiltonian using D-Wave schedules
# =========================
def HamiltonianDW(s, g=1, J=1):
    H_trans = np.kron(I, sx) + np.kron(sx, I)
    H_prob  = np.kron(sz, sz) + np.kron(sz, I) + np.kron(I, sz)
    return -g * dwaveA(s) * H_trans - J * dwaveB(s) * H_prob

# =========================
# Annealing grid
# =========================
s_vals = np.linspace(0, 1, 300)

eigenvalues = np.zeros((len(s_vals), 4))

# Diagonalise at each s
for i, s in enumerate(s_vals):
    eigenvalues[i] = np.linalg.eigvalsh(HamiltonianDW(s, g, J))

# =========================
# Plot spectrum
# =========================
plt.figure(figsize=(9,6))

for n in range(4):
    plt.plot(s_vals, eigenvalues[:, n], linewidth=2)

plt.xlabel("Anneal fraction s", fontsize=12)
plt.ylabel("Energy eigenvalues", fontsize=12)
plt.title("Adiabatic Spectrum using D-Wave Polynomial Annealing Schedule",
          fontsize=14, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# ============================
# Load D-Wave schedule
# ============================
file_path = "/content/09-1323A-A_Advantage2_system4_1_annealing_schedule.xlsx"
schedule = pd.read_excel(file_path, sheet_name="Standard-Annealing Schedule")

s_vals = schedule["s"].values
A_vals = schedule["A(s) (GHz)"].values
B_vals = schedule["B(s) (GHz)"].values
C_vals = schedule["C (normalized)"].values

# ============================
# Build interpolating functions
# ============================
# A(s), B(s) from s
A_of_s = interp1d(s_vals, A_vals, kind="cubic")
B_of_s = interp1d(s_vals, B_vals, kind="cubic")

# s(C) from C (C is monotonic increasing, so invert)
s_of_C = interp1d(C_vals, s_vals, kind="cubic")

# ============================
# Define time and control: C(t)
# ============================
T_anneal = 20.0  # total anneal time in microseconds (you can change this)
t = np.linspace(0.0, T_anneal, 1000)

# Assume hardware drives C linearly in time: C(t) = t / T linear drive
C_t = t / T_anneal              # goes from 0 to 1

# Corresponding logical s(t)
s_t = s_of_C(C_t)

# Time-dependent energies
A_t = A_of_s(s_t)
B_t = B_of_s(s_t)

# Instantaneous anneal speed ds/dt
ds_dt = np.gradient(s_t, C_t)

# ============================
# Visualisation in time
# ============================
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1) A(t), B(t)
axes[0,0].plot(C_t, A_t, label="A(s(t))", linewidth=2, color="crimson")
axes[0,0].plot(C_t, B_t, label="B(s(t))", linewidth=2, color="navy")
axes[0,0].set_xlabel("C(s(t)) (µs)")
axes[0,0].set_ylabel("Energy (GHz)")
axes[0,0].set_title("A(s(t)) and B(s(t))")
axes[0,0].grid(True, linestyle="--", alpha=0.6)
axes[0,0].legend()

# 2) C(t)
axes[0,1].plot(t, C_t, linewidth=2,color="darkgreen")
axes[0,1].set_xlabel("Time t (µs)")
axes[0,1].set_ylabel("C(s(t))")
axes[0,1].set_title("Hardware Control Function C(s(t)) (assumed to be linear)")
axes[0,1].grid(True, linestyle="--", alpha=0.6)

# 3) s(t)
axes[1,0].plot(t, s_t, linewidth=2, color="purple", label="s(t)")
axes[1,0].set_xlabel("Time t (µs)")
axes[1,0].set_ylabel("s(t)")
axes[1,0].set_title("Logical anneal fraction s(t)")
axes[1,0].grid(True, linestyle="--", alpha=0.6)

# 4) ds/dt
axes[1,1].plot(t, ds_dt, linewidth=2, color="orange")
axes[1,1].set_xlabel("Time t (µs)")
axes[1,1].set_ylabel("ds/dt")
axes[1,1].set_title("Instantaneous anneal speed ds/dt")
axes[1,1].grid(True, linestyle="--", alpha=0.6)

plt.suptitle(
    "Real D-Wave Annealing Dynamics",
    fontsize=16,
    fontweight="bold"
)
plt.tight_layout(rect=[0, 0, 1, 0.95])

plt.show()
